# Project 02 · Predicting loan default

**Benson M. Merita · MeritaAnalytics** · Tools: SQL (SQLite), Python (pandas, scikit-learn), Matplotlib

**Business question.** Can we rank loan applicants by their risk of default, and where should the approval cut-off sit to make the most money?

**Data.** The same synthetic lender database as Project 01: 560 loans, 1,000 customers and 20,000 wallet transactions. Synthetic data, generated with a fixed seed.

**How to read this notebook.** Each step has a short note on *what* it does and *why*. The model is only trusted after it beats simple baselines on loans it never saw.

| Step | What it answers |
|---|---|
| 1 | Build one row per loan with SQL |
| 2 | Remove leakage (information only known after the outcome) |
| 3 | Keep loans old enough to judge |
| 4 | Split into training and test sets |
| 5 | Set baselines the model must beat |
| 6 | Train and compare two models |
| 7 | Measure on the test set (ROC, AUC, KS) |
| 8 | Choose the approval cut-off by profit |
| 9 | Which features matter? |
| 10 | Fairness check |

## Setup

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, GREEN, GREY, DARK, AMBER = "#1F5FD6", "#2BB38A", "#9CA3AF", "#0B2A4A", "#B45309"
plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)

con = sqlite3.connect("../datasets/01-lending-credit-risk/sql/lender.db")

## Step 1 · Build one row per loan with SQL

**What:** join each loan to its customer, and summarise the customer's wallet activity in the **90 days before the loan was given out**: number of transactions, average amount and share of failed transactions.
**Why:** a model needs one row per decision. Using only activity *before* disbursement means the model uses what a lender would actually know on the day. `ORDER BY` fixes the row order, so the random train/test split in Step 4 gives the same result on every computer.

In [ ]:
query = """
WITH pre AS (
    SELECT l.loan_id,
           COUNT(t.txn_id)                         AS txn_90d,
           COALESCE(AVG(t.amount_ksh), 0)          AS avg_txn_ksh,
           COALESCE(AVG(t.status <> 'Success'), 0) AS failed_share
    FROM loans l
    LEFT JOIN transactions t
           ON t.customer_id = l.customer_id
          AND t.txn_date <  l.disbursement_date
          AND t.txn_date >= date(l.disbursement_date, '-90 days')
    GROUP BY l.loan_id
)
SELECT l.loan_id, l.product, l.principal_ksh, l.interest_rate_pct, l.term_months,
       l.disbursement_date, l.credit_score,
       c.age, c.gender, c.occupation, c.monthly_income_ksh, c.kyc_level, c.segment,
       pre.txn_90d, pre.avg_txn_ksh, pre.failed_share,
       (l.status = 'Defaulted') AS default_flag,
       l.status, l.days_past_due
FROM loans l
JOIN customers c ON c.customer_id = l.customer_id
JOIN pre         ON pre.loan_id   = l.loan_id
ORDER BY l.loan_id
"""
df = pd.read_sql(query, con, parse_dates=["disbursement_date"])

# affordability: monthly instalment as a share of monthly income
df["instalment_ratio"] = (df.principal_ksh * (1 + df.interest_rate_pct / 100 * df.term_months / 12)
                          / df.term_months / df.monthly_income_ksh)
print(df.shape, "| overall default rate:", round(df.default_flag.mean(), 3))
df[["product", "principal_ksh", "credit_score", "txn_90d", "instalment_ratio", "default_flag"]].head()

## Step 2 · Remove leakage

**What:** test how well single columns predict default. AUC runs from 0.5 (no better than a coin) to 1.0 (perfect).
**Why:** `days_past_due` gives a perfect score because it is only known *after* the loan goes bad. A model that uses it looks brilliant in testing and is useless on new applicants. It is removed, together with `status`.

In [ ]:
from sklearn.metrics import roc_auc_score

print("AUC using days_past_due (leakage):", round(roc_auc_score(df.default_flag, df.days_past_due), 3))
print("AUC using credit_score alone     :", round(roc_auc_score(df.default_flag, -df.credit_score), 3))

## Step 3 · Keep loans old enough to judge

**What:** keep loans observed for at least 270 days.
**Why:** a recent loan that has not defaulted *yet* is not a good loan. Counting it as good would teach the model the wrong lesson.

In [ ]:
AS_OF = pd.Timestamp("2025-12-31")
df["days_observed"] = (AS_OF - df.disbursement_date).dt.days
mature = df[df.days_observed >= 270].copy()
print("all loans:", len(df), "| mature loans kept:", len(mature),
      "| default rate:", round(mature.default_flag.mean(), 3))

## Step 4 · Split into training and test sets

**What:** keep 30% of loans aside as a test set the model never sees while it learns. `stratify` keeps the same default rate in both parts.
**Why:** the test set is our stand-in for future applicants. **Gender is deliberately not a model input**; it is kept aside only for the fairness check in Step 10.

In [ ]:
from sklearn.model_selection import train_test_split

num = ["principal_ksh", "interest_rate_pct", "term_months", "credit_score", "age",
       "monthly_income_ksh", "txn_90d", "avg_txn_ksh", "failed_share", "instalment_ratio"]
cat = ["product", "occupation", "kyc_level", "segment"]

X, y = mature[num + cat], mature.default_flag.astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
meta_te = mature.loc[X_te.index, ["gender", "segment"]]
print("train:", X_tr.shape, "default rate", round(y_tr.mean(), 3))
print("test :", X_te.shape, "default rate", round(y_te.mean(), 3))

## Step 5 · Baselines the model must beat

**What:** two simple benchmarks: (a) predict the same average risk for everyone, (b) rank by credit score alone.
**Why:** if a model cannot beat the credit score the lender already has, it adds nothing.

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="prior").fit(X_tr[num], y_tr)
print("same risk for everyone  : AUC", round(roc_auc_score(y_te, dummy.predict_proba(X_te[num])[:, 1]), 3))
print("credit score only       : AUC", round(roc_auc_score(y_te, -X_te.credit_score), 3))

## Step 6 · Train and compare two models

**What:** logistic regression (simple, easy to explain) against gradient boosting (more flexible). Both are scored with **5-fold cross-validation** on the training set, then once on the test set.
**Why:** cross-validation shows how stable each model is; the test set shows how it does on unseen loans. The text columns are one-hot encoded so the models can use them.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

prep_lr = ColumnTransformer([("num", StandardScaler(), num),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
prep_gb = ColumnTransformer([("num", "passthrough", num),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
models = {
    "logistic": Pipeline([("prep", prep_lr), ("model", LogisticRegression(max_iter=1000))]),
    "boosting": Pipeline([("prep", prep_gb), ("model", HistGradientBoostingClassifier(
        max_depth=3, learning_rate=0.05, max_iter=150, random_state=42))]),
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for name, model in models.items():
    cv_auc = cross_val_score(model, X_tr, y_tr, cv=cv, scoring="roc_auc")
    model.fit(X_tr, y_tr)
    test_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    rows.append([name, cv_auc.mean(), cv_auc.std(), test_auc])
pd.DataFrame(rows, columns=["model", "cv_auc_mean", "cv_auc_std", "test_auc"]).round(3)

Both models beat the credit score alone. The simpler **logistic regression** does best on the test set, so it is the one we keep: it is also easier to explain to a credit committee.

## Step 7 · Test-set ROC curves and KS

**What:** the ROC curve shows, for every possible cut-off, the share of defaults caught against the share of good loans wrongly flagged. **KS** is the largest gap between the two, a standard scorecard measure in credit risk.

In [ ]:
from sklearn.metrics import roc_curve

def ks_stat(y_true, p):
    fpr, tpr, _ = roc_curve(y_true, p)
    return float((tpr - fpr).max())

fig, ax = plt.subplots()
for name, colour in [("logistic", BLUE), ("boosting", GREEN)]:
    p = models[name].predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, p)
    ax.plot(fpr, tpr, color=colour, lw=2,
            label=f"{name}: AUC {roc_auc_score(y_te, p):.2f}, KS {ks_stat(y_te, p):.2f}")
ax.plot([0, 1], [0, 1], color=GREY, ls="--", label="no skill")
ax.set_xlabel("False positive rate (good loans flagged as risky)")
ax.set_ylabel("True positive rate (defaults caught)")
ax.set_title("Test-set ROC curves"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

## Step 8 · Choose the approval cut-off by profit

**What:** approve an applicant when predicted risk is below a cut-off. A good loan earns 8% of principal and a default loses 60% (stated assumptions). The cut-off is chosen on the **training** data only, then applied once to the test set.
**Why:** choosing the cut-off on the test set would be cheating. The business cares about shillings, not accuracy.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

GAIN, LOSS = 0.08, 0.60          # assumptions, as a share of principal
final = models["logistic"]
oof = cross_val_predict(final, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]

def profit(p, y_true, principal, cut):
    approve = p < cut
    value = np.where(y_true == 0, GAIN, -LOSS) * principal
    return float(value[approve].sum()), float(approve.mean())

cuts = np.linspace(0.05, 0.60, 56)
train_profit = [profit(oof, y_tr.to_numpy(), X_tr.principal_ksh.to_numpy(), c)[0] for c in cuts]
best_cut = float(cuts[int(np.argmax(train_profit))])
print("best cut-off (chosen on training folds):", round(best_cut, 2))

p_te = final.predict_proba(X_te)[:, 1]
decline = (p_te >= best_cut).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, decline).ravel()
p_all, _ = profit(p_te, y_te.to_numpy(), X_te.principal_ksh.to_numpy(), 9)
p_cut, appr = profit(p_te, y_te.to_numpy(), X_te.principal_ksh.to_numpy(), best_cut)
print(f"test set: declined {decline.sum()} of {len(decline)} | defaults caught {tp} of {tp + fn} "
      f"| good loans declined {fp}")
print(f"test profit: approve everyone Ksh {p_all:,.0f} | with the model Ksh {p_cut:,.0f} "
      f"(approval rate {appr:.0%})")

## Step 9 · Which features matter?

**What:** *permutation importance* shuffles one column at a time and measures how much the test AUC drops. A bigger drop means the model relies on that column more.

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(final, X_te, y_te, scoring="roc_auc", n_repeats=15, random_state=42)
imp = pd.Series(pi.importances_mean, index=X_te.columns).sort_values()
fig, ax = plt.subplots()
ax.barh(imp.index, imp.values, color=[AMBER if v == imp.max() else BLUE for v in imp])
ax.axvline(0, color=DARK, lw=1)
ax.set_xlabel("Drop in test AUC when the column is shuffled"); ax.set_title("Permutation importance")
plt.tight_layout(); plt.show()
print("most important feature:", imp.idxmax())

## Step 10 · Fairness check

**What:** compare approval rates and actual default rates by gender and customer segment at the chosen cut-off.
**Why:** a model can be accurate and still treat groups unequally. A common rule of thumb flags a ratio below 0.80 between the lowest and highest approval rates for review.

In [ ]:
audit = meta_te.assign(actual_default=y_te.to_numpy(), declined=decline)
for col in ["gender", "segment"]:
    g = audit.groupby(col)
    display(pd.DataFrame({"loans": g.size(), "approval_rate": 1 - g.declined.mean(),
                          "actual_default": g.actual_default.mean()}).round(3))
ar = 1 - audit.groupby("gender").declined.mean()
print("approval-rate ratio, lowest / highest (gender):", round(ar.min() / ar.max(), 2))

## Key findings and recommendations

- The **logistic model** reaches a test AUC of 0.84 and a KS of 0.56, clearly above the credit score alone (0.74). It also beat gradient boosting in cross-validation, so the simpler model is the right choice.
- The **instalment-to-income ratio** is the strongest predictor, the same signal found in Project 01.
- At the profit-based cut-off the model declines 38 of 122 test applicants and catches 18 of 25 defaults, turning a Ksh 1.59M loss into a Ksh 462k profit.
- **Fairness:** women are approved at 0.89 times the rate of men, although both groups default at about the same rate in this sample. That passes the common four-fifths rule, but the gap should be explained before any real use. The Mass segment's 17% approval rate (its loans default at 54%) needs a policy decision.

**Limitations.** Only 404 usable loans and 122 test loans, so the test AUC moves by several points with a different random split; the cross-validated AUC (0.84 ± 0.07) is the steadier figure. The data is synthetic and active loans are treated as good, so these results show the method, not production performance. Retrain once more loans mature.